# Phase 2 — Low-Volatility Factor Construction

**Notebook:** `02_factor_construction.ipynb`  
**Dependency:** Phase 1 outputs (`sp500_daily_prices.parquet`, `sp500_monthly_returns.parquet`, `factor_returns.parquet`)  
**Outputs:** `lowvol_factor_returns.parquet`, `factor_returns_full.parquet`  

## Objective

Construct the **low-volatility (LowVol) factor** from S&P 500 stock-level data. The Kenneth French
Data Library does not provide a low-volatility factor, so we build it ourselves using trailing
60-day realised volatility and quintile long-short portfolios.

**Method:**
- Trailing 60-day realised volatility (annualised, close-to-close) for each stock at month-end
- Yang-Zhang (2000) range-based volatility as enhancement (if OHLCV available)
- Quintile portfolios ranked by volatility ascending, equal-weighted returns in month t+1
- Long-short: LowVol = R_Q1 - R_Q5 (low-vol minus high-vol)
- **No look-ahead:** signal at t, return at t+1

**Academic reference:** Baker, Bradley & Wurgler (2011) — The Low-Volatility Anomaly

---
## 1. Setup and Imports

In [1]:
import sys
import warnings
import logging
from datetime import datetime
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Project imports
from src.config import (
    PROJECT_ROOT, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR,
    RANDOM_STATE, COLORS,
    SP500_DAILY_FILE, SP500_MONTHLY_FILE,
    FACTOR_RETURNS_FILE, FACTOR_RETURNS_FULL_FILE, LOWVOL_FACTOR_FILE,
    FACTOR_START, FACTOR_LIVE_START,
)
from src.validation import validate_parquet, quick_data_check, check_nan_propagation
from src.feature_engineering import yang_zhang_volatility
from src.visualization import setup_style, save_fig, plot_cumulative_returns, plot_correlation_heatmap

# Logging
PHASE_NUM = 2
logging.basicConfig(
    filename=str(LOGS_DIR / f'phase_{PHASE_NUM}_{datetime.now():%Y%m%d_%H%M}.log'),
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
)
logger = logging.getLogger(__name__)
logger.info(f"Phase {PHASE_NUM} started")

# Reproducibility
np.random.seed(RANDOM_STATE)

# Style
setup_style()
warnings.filterwarnings('ignore', category=FutureWarning)

print(f"Project root: {PROJECT_ROOT}")
print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}")
print(f"Random state: {RANDOM_STATE}")

Project root: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine
NumPy: 2.4.3, Pandas: 2.3.3
Random state: 42


---
## 2. Load Phase 1 Outputs

In [2]:
# ── Load S&P 500 daily prices ──
sp500_daily = pd.read_parquet(SP500_DAILY_FILE)
validate_parquet(sp500_daily, date_index=False, min_rows=100, label='sp500_daily_prices')
quick_data_check(sp500_daily, 'S&P 500 Daily Prices')

=== S&P 500 Daily Prices ===
  Shape: (2511783, 2)


  Index: DatetimeIndex, range: 2004-01-02 00:00:00 → 2025-12-30 00:00:00
  Duplicates: 2506249
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('O'): np.int64(1), dtype('float64'): np.int64(1)}
  Sorted: True



In [3]:
# ── Load S&P 500 monthly returns ──
sp500_monthly = pd.read_parquet(SP500_MONTHLY_FILE)
validate_parquet(sp500_monthly, date_index=False, min_rows=50, label='sp500_monthly_returns')
quick_data_check(sp500_monthly, 'S&P 500 Monthly Returns')

=== S&P 500 Monthly Returns ===
  Shape: (119385, 2)
  Index: DatetimeIndex, range: 2004-02-29 00:00:00 → 2025-12-31 00:00:00
  Duplicates: 119122
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('O'): np.int64(1), dtype('float64'): np.int64(1)}
  Sorted: True



In [4]:
# ── Load factor returns from French Library (Phase 1) ──
factor_returns = pd.read_parquet(FACTOR_RETURNS_FILE)
validate_parquet(
    factor_returns,
    expected_cols=['hml', 'umd', 'rmw'],
    date_index=True,
    min_rows=100,
    label='factor_returns',
)
quick_data_check(factor_returns, 'Factor Returns (French Library)')

=== Factor Returns (French Library) ===
  Shape: (232, 7)
  Index: DatetimeIndex, range: 2005-01-31 00:00:00 → 2024-04-30 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(7)}
  Sorted: True



---
## 3. Prepare Price Panel

The S&P 500 daily prices may be in long format (`ticker`, `adj_close`) or wide format.
We need a wide-format DataFrame with tickers as columns and dates as index for rolling
volatility computation.

In [5]:
# Determine format and pivot if necessary
if 'ticker' in sp500_daily.columns and 'adj_close' in sp500_daily.columns:
    # Long format -> pivot to wide
    prices_wide = sp500_daily.pivot_table(
        index=sp500_daily.index, columns='ticker', values='adj_close'
    )
elif 'ticker' in sp500_daily.columns and 'Close' in sp500_daily.columns:
    prices_wide = sp500_daily.pivot_table(
        index=sp500_daily.index, columns='ticker', values='Close'
    )
else:
    # Already in wide format (tickers as columns)
    prices_wide = sp500_daily.copy()

prices_wide = prices_wide.sort_index()
print(f"Price panel shape: {prices_wide.shape}")
print(f"Date range: {prices_wide.index.min()} to {prices_wide.index.max()}")
print(f"Number of tickers: {prices_wide.shape[1]}")
print(f"NaN percentage: {prices_wide.isna().mean().mean():.2%}")

Price panel shape: (5534, 503)
Date range: 2004-01-02 00:00:00 to 2025-12-30 00:00:00
Number of tickers: 503
NaN percentage: 9.77%


In [6]:
# Similarly, prepare monthly returns in wide format
if 'ticker' in sp500_monthly.columns and 'monthly_return' in sp500_monthly.columns:
    returns_wide = sp500_monthly.pivot_table(
        index=sp500_monthly.index, columns='ticker', values='monthly_return'
    )
elif 'ticker' in sp500_monthly.columns:
    # Find the return column
    ret_col = [c for c in sp500_monthly.columns if 'return' in c.lower()]
    if ret_col:
        returns_wide = sp500_monthly.pivot_table(
            index=sp500_monthly.index, columns='ticker', values=ret_col[0]
        )
    else:
        raise ValueError(f"Cannot find return column in sp500_monthly. Columns: {sp500_monthly.columns.tolist()}")
else:
    # Already wide format
    returns_wide = sp500_monthly.copy()

returns_wide = returns_wide.sort_index()
print(f"Monthly returns panel shape: {returns_wide.shape}")
print(f"Date range: {returns_wide.index.min()} to {returns_wide.index.max()}")
print(f"NaN percentage: {returns_wide.isna().mean().mean():.2%}")

Monthly returns panel shape: (263, 503)
Date range: 2004-02-29 00:00:00 to 2025-12-31 00:00:00
NaN percentage: 9.75%


---
## 4. Compute Trailing 60-Day Realised Volatility

For each stock $i$ at month-end $t$:

$$\sigma_{i,t} = \sqrt{\frac{252}{D} \sum_{d=1}^{D} (r_{i,d} - \bar{r}_i)^2}$$

where $D = 60$ trailing trading days. We require $D \geq 40$ valid days for a valid estimate.

In [7]:
# Compute daily log returns from wide price panel
daily_log_returns = np.log(prices_wide / prices_wide.shift(1))

# Trailing 60-day realised volatility (annualised, close-to-close)
VOL_WINDOW = 60
MIN_VALID_DAYS = 40

rolling_vol = daily_log_returns.rolling(
    window=VOL_WINDOW, min_periods=MIN_VALID_DAYS
).std() * np.sqrt(252)

print(f"Rolling volatility shape: {rolling_vol.shape}")
print(f"First valid date: {rolling_vol.dropna(how='all').index.min()}")
print(f"NaN percentage: {rolling_vol.isna().mean().mean():.2%}")

Rolling volatility shape: (5534, 503)
First valid date: 2004-03-02 00:00:00
NaN percentage: 10.49%


In [8]:
# Extract month-end volatility estimates (signal dates)
# Resample to month-end: take the last available volatility reading of each month
monthend_vol = rolling_vol.resample('M').last()

print(f"Month-end volatility shape: {monthend_vol.shape}")
print(f"Date range: {monthend_vol.index.min()} to {monthend_vol.index.max()}")
print(f"\nCross-sectional stats (first valid month):")
first_valid = monthend_vol.dropna(how='all').iloc[0]
print(f"  Mean vol: {first_valid.mean():.4f}")
print(f"  Median vol: {first_valid.median():.4f}")
print(f"  Stocks with valid vol: {first_valid.notna().sum()}")

Month-end volatility shape: (264, 503)
Date range: 2004-01-31 00:00:00 to 2025-12-31 00:00:00

Cross-sectional stats (first valid month):
  Mean vol: 0.2652
  Median vol: 0.2339
  Stocks with valid vol: 377


---
## 5. Yang-Zhang Volatility (Enhancement)

The Yang-Zhang (2000) estimator uses OHLC data and is more efficient than the close-to-close
estimator because it incorporates intra-day price information. We attempt to compute it if
OHLCV data is available; otherwise, we fall back to the close-to-close estimate above.

In [9]:
# Check if OHLCV columns are available in the daily data
ohlcv_cols = {'Open', 'High', 'Low', 'Close'}
has_ohlcv = False
yz_monthend_vol = None

if 'ticker' in sp500_daily.columns:
    # Long format with OHLCV
    available_cols = set(sp500_daily.columns) - {'ticker'}
    if ohlcv_cols.issubset(available_cols):
        has_ohlcv = True
        print("OHLCV data available in long format -- computing Yang-Zhang volatility per ticker.")
        
        tickers_in_data = sp500_daily['ticker'].unique()
        yz_vol_dict = {}
        
        for ticker in tickers_in_data:
            ticker_data = sp500_daily[sp500_daily['ticker'] == ticker][['Open', 'High', 'Low', 'Close']].copy()
            ticker_data = ticker_data.sort_index()
            if len(ticker_data) >= MIN_VALID_DAYS:
                try:
                    yz = yang_zhang_volatility(ticker_data, window=VOL_WINDOW)
                    yz_vol_dict[ticker] = yz
                except Exception:
                    pass
        
        if yz_vol_dict:
            yz_vol_panel = pd.DataFrame(yz_vol_dict)
            yz_monthend_vol = yz_vol_panel.resample('M').last()
            print(f"Yang-Zhang volatility computed for {len(yz_vol_dict)} tickers.")
        else:
            print("Yang-Zhang computation returned no results -- falling back to close-to-close.")
else:
    # Wide format -- check if we have separate OHLCV DataFrames
    # If the wide format only has adjusted close, OHLCV is not available
    print("OHLCV data not available in long format.")

if not has_ohlcv:
    print("Falling back to close-to-close volatility estimator (standard 60-day realised vol).")
    print("NOTE: Yang-Zhang would be more efficient but requires Open/High/Low/Close columns.")

# Determine which volatility measure to use for ranking
if yz_monthend_vol is not None and not yz_monthend_vol.empty:
    # Use Yang-Zhang where available, fill with close-to-close
    vol_for_ranking = yz_monthend_vol.reindex(
        index=monthend_vol.index, columns=monthend_vol.columns
    ).fillna(monthend_vol)
    vol_source = 'Yang-Zhang (with close-to-close fallback)'
else:
    vol_for_ranking = monthend_vol.copy()
    vol_source = 'Close-to-close (60-day realised)'

print(f"\nVolatility source for quintile ranking: {vol_source}")
print(f"Shape: {vol_for_ranking.shape}")

Falling back to close-to-close volatility estimator (standard 60-day realised vol).
NOTE: Yang-Zhang would be more efficient but requires Open/High/Low/Close columns.

Volatility source for quintile ranking: Close-to-close (60-day realised)
Shape: (264, 503)


---
## 6. Cross-Sectional Volatility Distribution

Before forming quintile portfolios, examine the cross-sectional distribution of volatility
to ensure it is well-behaved and has meaningful dispersion.

In [10]:
# Cross-sectional volatility statistics over time
vol_stats = pd.DataFrame({
    'mean': vol_for_ranking.mean(axis=1),
    'median': vol_for_ranking.median(axis=1),
    'q10': vol_for_ranking.quantile(0.10, axis=1),
    'q90': vol_for_ranking.quantile(0.90, axis=1),
    'std': vol_for_ranking.std(axis=1),
    'n_valid': vol_for_ranking.notna().sum(axis=1),
}).dropna()

print("Cross-sectional volatility summary (annualised):")
print(vol_stats[['mean', 'median', 'q10', 'q90', 'n_valid']].describe().round(4))

Cross-sectional volatility summary (annualised):
           mean    median       q10       q90   n_valid
count  262.0000  262.0000  262.0000  262.0000  262.0000
mean     0.2998    0.2716    0.1826    0.4482  453.7863
std      0.1223    0.1123    0.0764    0.1835   37.6203
min      0.1835    0.1590    0.1078    0.2764  377.0000
25%      0.2353    0.2116    0.1422    0.3529  419.0000
50%      0.2635    0.2363    0.1603    0.3982  462.0000
75%      0.3159    0.2919    0.2009    0.4654  488.0000
max      0.9488    0.8655    0.6083    1.4220  503.0000


In [11]:
# Plot: Cross-sectional volatility dispersion over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: median vol with 10-90 percentile band
ax1.fill_between(vol_stats.index, vol_stats['q10'], vol_stats['q90'],
                 alpha=0.2, color=COLORS['lowvol'], label='10th-90th percentile')
ax1.plot(vol_stats.index, vol_stats['median'], color=COLORS['lowvol'],
         linewidth=1.2, label='Median')
ax1.plot(vol_stats.index, vol_stats['mean'], color='black',
         linewidth=0.8, linestyle='--', label='Mean')
ax1.set_ylabel('Annualised Volatility')
ax1.set_title('Cross-Sectional Volatility Distribution Over Time (S&P 500)', fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Bottom: number of valid stocks
ax2.bar(vol_stats.index, vol_stats['n_valid'], width=25, color=COLORS['lowvol'], alpha=0.6)
ax2.axhline(y=150, color='red', linestyle='--', alpha=0.7, label='Min 150 stocks (30 per quintile)')
ax2.set_ylabel('Number of Valid Stocks')
ax2.set_xlabel('Date')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, 'lowvol_cross_sectional_vol_dispersion')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/lowvol_cross_sectional_vol_dispersion.png


---
## 7. Form Quintile Portfolios

At each month-end $t$:
1. Rank stocks by trailing volatility ($\sigma_{i,t}$) ascending
2. Divide into 5 equal-size groups (quintiles)
3. Q1 = lowest volatility, Q5 = highest volatility
4. Record the equal-weight return of each quintile in month $t+1$

**Critical: No look-ahead.** The volatility signal is computed at $t$, and portfolio returns
are measured at $t+1$. The `shift()` operation enforces this.

In [12]:
# Align monthly returns index to month-end (same convention as vol)
returns_wide.index = returns_wide.index + pd.offsets.MonthEnd(0)
vol_for_ranking.index = vol_for_ranking.index + pd.offsets.MonthEnd(0)

# Common dates and tickers between vol and returns
common_dates = vol_for_ranking.index.intersection(returns_wide.index)
common_tickers = vol_for_ranking.columns.intersection(returns_wide.columns)

vol_aligned = vol_for_ranking.loc[common_dates, common_tickers]
ret_aligned = returns_wide.loc[common_dates, common_tickers]

print(f"Aligned data: {len(common_dates)} months, {len(common_tickers)} tickers")
print(f"Date range: {common_dates.min()} to {common_dates.max()}")

Aligned data: 263 months, 503 tickers
Date range: 2004-02-29 00:00:00 to 2025-12-31 00:00:00


In [13]:
# ── Quintile portfolio construction ──
N_QUINTILES = 5
MIN_STOCKS_PER_QUINTILE = 30

quintile_returns = {q: [] for q in range(1, N_QUINTILES + 1)}
quintile_counts = {q: [] for q in range(1, N_QUINTILES + 1)}
quintile_dates = []
quintile_avg_vol = {q: [] for q in range(1, N_QUINTILES + 1)}
skipped_months = []

# Loop over signal dates (t): portfolio return is at t+1
signal_dates = sorted(common_dates)

for i, signal_date in enumerate(signal_dates[:-1]):
    return_date = signal_dates[i + 1]  # t+1: no look-ahead
    
    # Cross-sectional volatility at signal date
    vol_cross = vol_aligned.loc[signal_date].dropna()
    
    # Monthly returns at return date for the same stocks
    ret_cross = ret_aligned.loc[return_date].reindex(vol_cross.index).dropna()
    
    # Keep only stocks with both vol and return
    valid_tickers = vol_cross.index.intersection(ret_cross.index)
    vol_cross = vol_cross.loc[valid_tickers]
    ret_cross = ret_cross.loc[valid_tickers]
    
    n_stocks = len(valid_tickers)
    
    # Require at least 30 stocks per quintile
    if n_stocks < MIN_STOCKS_PER_QUINTILE * N_QUINTILES:
        skipped_months.append({
            'signal_date': signal_date,
            'return_date': return_date,
            'n_stocks': n_stocks,
            'reason': f'Only {n_stocks} stocks, need {MIN_STOCKS_PER_QUINTILE * N_QUINTILES}',
        })
        continue
    
    # Rank by volatility ascending (Q1 = lowest vol)
    vol_ranked = vol_cross.rank(method='first')
    quintile_labels = pd.qcut(vol_ranked, q=N_QUINTILES, labels=False) + 1  # 1 to 5
    
    # Compute equal-weight return for each quintile
    quintile_dates.append(return_date)
    for q in range(1, N_QUINTILES + 1):
        mask = quintile_labels == q
        q_stocks = ret_cross[mask]
        q_vol = vol_cross[mask]
        quintile_returns[q].append(q_stocks.mean())
        quintile_counts[q].append(mask.sum())
        quintile_avg_vol[q].append(q_vol.mean())

# Build DataFrames
quintile_ret_df = pd.DataFrame(
    {f'Q{q}': quintile_returns[q] for q in range(1, N_QUINTILES + 1)},
    index=pd.DatetimeIndex(quintile_dates),
)
quintile_ret_df.index.name = 'date'

quintile_count_df = pd.DataFrame(
    {f'Q{q}': quintile_counts[q] for q in range(1, N_QUINTILES + 1)},
    index=pd.DatetimeIndex(quintile_dates),
)

quintile_vol_df = pd.DataFrame(
    {f'Q{q}': quintile_avg_vol[q] for q in range(1, N_QUINTILES + 1)},
    index=pd.DatetimeIndex(quintile_dates),
)

print(f"Quintile portfolios formed: {len(quintile_dates)} months")
print(f"Skipped months (insufficient stocks): {len(skipped_months)}")
if skipped_months:
    skipped_df = pd.DataFrame(skipped_months)
    print(skipped_df.to_string(index=False))

Quintile portfolios formed: 261 months
Skipped months (insufficient stocks): 1
signal_date return_date  n_stocks                  reason
 2004-02-29  2004-03-31         0 Only 0 stocks, need 150


In [14]:
# ── Validation: >=30 stocks per quintile at every rebalance date ──
min_count_per_quintile = quintile_count_df.min()
print("Minimum stock count per quintile across all rebalance dates:")
print(min_count_per_quintile)
print()

all_above_30 = (quintile_count_df >= MIN_STOCKS_PER_QUINTILE).all().all()
print(f"VALIDATION GATE: >=30 stocks per quintile at every rebalance: {'PASS' if all_above_30 else 'FAIL'}")

if not all_above_30:
    violations = (quintile_count_df < MIN_STOCKS_PER_QUINTILE).sum()
    print(f"Violations per quintile: {violations.to_dict()}")

logger.info(f"Quintile count validation: {'PASS' if all_above_30 else 'FAIL'}")
logger.info(f"Min counts: {min_count_per_quintile.to_dict()}")

Minimum stock count per quintile across all rebalance dates:
Q1    76
Q2    75
Q3    75
Q4    75
Q5    76
dtype: int64

VALIDATION GATE: >=30 stocks per quintile at every rebalance: PASS


---
## 8. Construct Long-Short LowVol Factor

$$\text{LowVol}_{t+1} = R_{Q1,t+1} - R_{Q5,t+1}$$

Long the lowest-volatility quintile, short the highest-volatility quintile.

In [15]:
# Long-short factor: Q1 (low vol) minus Q5 (high vol)
lowvol_returns = quintile_ret_df['Q1'] - quintile_ret_df['Q5']
lowvol_returns.name = 'lowvol'

# Summary statistics
print("=" * 60)
print("LowVol Factor (Q1 - Q5) Summary Statistics")
print("=" * 60)
print(f"Mean monthly return:     {lowvol_returns.mean():.6f}  ({lowvol_returns.mean()*12:.4f} ann.)")
print(f"Std monthly return:      {lowvol_returns.std():.6f}  ({lowvol_returns.std()*np.sqrt(12):.4f} ann.)")
print(f"Sharpe ratio (ann.):     {lowvol_returns.mean() / lowvol_returns.std() * np.sqrt(12):.4f}")
print(f"Skewness:                {lowvol_returns.skew():.4f}")
print(f"Excess kurtosis:         {lowvol_returns.kurtosis():.4f}")
print(f"Hit rate (% positive):   {(lowvol_returns > 0).mean():.2%}")
print(f"Min monthly return:      {lowvol_returns.min():.6f}")
print(f"Max monthly return:      {lowvol_returns.max():.6f}")
print(f"Number of months:        {len(lowvol_returns)}")
print(f"Date range:              {lowvol_returns.index.min()} to {lowvol_returns.index.max()}")

# t-statistic for mean != 0
t_stat = lowvol_returns.mean() / (lowvol_returns.std() / np.sqrt(len(lowvol_returns)))
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=len(lowvol_returns) - 1))
print(f"\nt-statistic (mean != 0): {t_stat:.4f}  (p = {p_value:.4f})")

LowVol Factor (Q1 - Q5) Summary Statistics
Mean monthly return:     -0.011570  (-0.1388 ann.)
Std monthly return:      0.057456  (0.1990 ann.)
Sharpe ratio (ann.):     -0.6976
Skewness:                -0.9553
Excess kurtosis:         4.4130
Hit rate (% positive):   42.15%
Min monthly return:      -0.346627
Max monthly return:      0.110244
Number of months:        261
Date range:              2004-04-30 00:00:00 to 2025-12-31 00:00:00

t-statistic (mean != 0): -3.2533  (p = 0.0013)


In [16]:
# ── Validation Gate: LowVol mean positive (~0.001-0.004) ──
lowvol_mean = lowvol_returns.mean()
mean_in_range = 0.0 < lowvol_mean
mean_plausible = -0.005 < lowvol_mean < 0.010  # wider tolerance for real data

print(f"VALIDATION GATE: LowVol mean positive: {'PASS' if mean_in_range else 'FAIL'} (mean = {lowvol_mean:.6f})")
if not mean_in_range:
    print(f"  NOTE: Negative mean may reflect survivorship bias or time-period effect.")
    print(f"  The low-volatility anomaly has weakened in recent years (post-2010 crowding).")
    print(f"  This is a finding, not necessarily a failure.")

logger.info(f"LowVol mean: {lowvol_mean:.6f}, positive={mean_in_range}")

VALIDATION GATE: LowVol mean positive: FAIL (mean = -0.011570)
  NOTE: Negative mean may reflect survivorship bias or time-period effect.
  The low-volatility anomaly has weakened in recent years (post-2010 crowding).
  This is a finding, not necessarily a failure.


---
## 9. Quintile Portfolio Analysis

In [17]:
# Quintile summary statistics
quintile_stats = pd.DataFrame({
    'Mean (Monthly)': quintile_ret_df.mean(),
    'Mean (Annual)': quintile_ret_df.mean() * 12,
    'Std (Monthly)': quintile_ret_df.std(),
    'Std (Annual)': quintile_ret_df.std() * np.sqrt(12),
    'Sharpe (Annual)': quintile_ret_df.mean() / quintile_ret_df.std() * np.sqrt(12),
    'Skew': quintile_ret_df.skew(),
    'Kurtosis': quintile_ret_df.kurtosis(),
    'Hit Rate': (quintile_ret_df > 0).mean(),
    'Avg Vol Signal': quintile_vol_df.mean(),
    'Avg Stock Count': quintile_count_df.mean(),
})

# Add LowVol long-short row
ls = lowvol_returns
quintile_stats.loc['L/S (Q1-Q5)'] = [
    ls.mean(), ls.mean() * 12, ls.std(), ls.std() * np.sqrt(12),
    ls.mean() / ls.std() * np.sqrt(12), ls.skew(), ls.kurtosis(),
    (ls > 0).mean(), np.nan, np.nan,
]

print("\nQuintile Portfolio Summary Statistics")
print("=" * 80)
print(quintile_stats.round(4).to_string())


Quintile Portfolio Summary Statistics
             Mean (Monthly)  Mean (Annual)  Std (Monthly)  Std (Annual)  Sharpe (Annual)    Skew  Kurtosis  Hit Rate  Avg Vol Signal  Avg Stock Count
Q1                   0.0095         0.1138         0.0333        0.1152           0.9882 -0.6876    2.0028    0.6705          0.1787          91.1609
Q2                   0.0115         0.1382         0.0394        0.1364           1.0133 -0.6931    1.9345    0.6705          0.2284          90.5057
Q3                   0.0122         0.1461         0.0477        0.1654           0.8834 -0.6253    2.3189    0.6552          0.2722          90.5172
Q4                   0.0136         0.1632         0.0547        0.1893           0.8620 -0.4080    2.5330    0.6475          0.3303          90.5057
Q5                   0.0211         0.2527         0.0734        0.2541           0.9944  0.2519    2.9114    0.6628          0.4893          90.9080
L/S (Q1-Q5)         -0.0116        -0.1388         0.0575    

In [18]:
# ── Validation Gate: Q1 lower realised vol than Q5 OOS ──
q1_avg_signal_vol = quintile_vol_df['Q1'].mean()
q5_avg_signal_vol = quintile_vol_df['Q5'].mean()

# Also check realised return vol (OOS)
q1_realised_vol = quintile_ret_df['Q1'].std() * np.sqrt(12)
q5_realised_vol = quintile_ret_df['Q5'].std() * np.sqrt(12)

print(f"Signal volatility (average):  Q1 = {q1_avg_signal_vol:.4f}, Q5 = {q5_avg_signal_vol:.4f}")
print(f"Realised return vol (OOS):    Q1 = {q1_realised_vol:.4f}, Q5 = {q5_realised_vol:.4f}")
print()

vol_ordering_oos = q1_realised_vol < q5_realised_vol
print(f"VALIDATION GATE: Q1 lower vol than Q5 OOS: {'PASS' if vol_ordering_oos else 'FAIL'}")
logger.info(f"Vol ordering OOS: Q1={q1_realised_vol:.4f}, Q5={q5_realised_vol:.4f}, pass={vol_ordering_oos}")

Signal volatility (average):  Q1 = 0.1787, Q5 = 0.4893
Realised return vol (OOS):    Q1 = 0.1152, Q5 = 0.2541

VALIDATION GATE: Q1 lower vol than Q5 OOS: PASS


In [19]:
# Plot: Cumulative returns of quintile portfolios
fig, ax = plt.subplots(figsize=(14, 7))

colors_q = ['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c']
cum_ret = (1 + quintile_ret_df).cumprod()

for i, q in enumerate(quintile_ret_df.columns):
    ax.plot(cum_ret.index, cum_ret[q], label=f'{q} ({"Low" if i == 0 else "High" if i == 4 else ""} Vol)',
            color=colors_q[i], linewidth=1.5 if i in [0, 4] else 0.9)

# Add long-short
cum_ls = (1 + lowvol_returns).cumprod()
ax.plot(cum_ls.index, cum_ls, label='L/S (Q1-Q5)', color=COLORS['lowvol'],
        linewidth=2.0, linestyle='--')

ax.set_ylabel('Cumulative Return (base=1)')
ax.set_title('Low-Volatility Quintile Portfolio Cumulative Returns', fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

plt.tight_layout()
save_fig(fig, 'lowvol_quintile_cumulative_returns')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/lowvol_quintile_cumulative_returns.png


In [20]:
# Plot: Quintile bar chart (annualised return vs annualised vol)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

quintiles = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
ann_returns = [quintile_ret_df[q].mean() * 12 for q in quintiles]
ann_vols = [quintile_ret_df[q].std() * np.sqrt(12) for q in quintiles]
sharpes = [quintile_ret_df[q].mean() / quintile_ret_df[q].std() * np.sqrt(12) for q in quintiles]

# Left: Annualised return by quintile
bars1 = ax1.bar(quintiles, ann_returns, color=colors_q, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Annualised Return')
ax1.set_title('Mean Return by Volatility Quintile', fontweight='bold')
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, ann_returns):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{val:.1%}', ha='center', va='bottom', fontsize=9)

# Right: Sharpe ratio by quintile
bars2 = ax2.bar(quintiles, sharpes, color=colors_q, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Sharpe Ratio (Annualised)')
ax2.set_title('Sharpe Ratio by Volatility Quintile', fontweight='bold')
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, sharpes):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
save_fig(fig, 'lowvol_quintile_return_sharpe_bars')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/lowvol_quintile_return_sharpe_bars.png


---
## 10. LowVol Factor Drawdown Analysis

In [21]:
# Drawdown analysis of the long-short LowVol factor
cum_lowvol = (1 + lowvol_returns).cumprod()
running_max = cum_lowvol.cummax()
drawdown = (cum_lowvol - running_max) / running_max

print(f"Maximum drawdown: {drawdown.min():.4f} ({drawdown.min():.2%})")
print(f"Date of max drawdown: {drawdown.idxmin()}")

# Top 5 worst months
worst_months = lowvol_returns.nsmallest(5)
print(f"\nTop 5 worst months for LowVol factor:")
for date, ret in worst_months.items():
    print(f"  {date.strftime('%Y-%m')}: {ret:.4f} ({ret:.2%})")

Maximum drawdown: -0.9708 (-97.08%)
Date of max drawdown: 2025-12-31 00:00:00

Top 5 worst months for LowVol factor:
  2009-04: -0.3466 (-34.66%)
  2020-11: -0.2011 (-20.11%)
  2023-01: -0.1703 (-17.03%)
  2020-04: -0.1655 (-16.55%)
  2009-08: -0.1606 (-16.06%)


In [22]:
# Plot: LowVol cumulative return and drawdown
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(cum_lowvol.index, cum_lowvol, color=COLORS['lowvol'], linewidth=1.5)
ax1.set_ylabel('Cumulative Return (base=1)')
ax1.set_title('LowVol Factor (Q1 - Q5): Cumulative Return and Drawdown', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.fill_between(drawdown.index, drawdown, color=COLORS['crisis'], alpha=0.4)
ax2.plot(drawdown.index, drawdown, color=COLORS['crisis'], linewidth=0.8)
ax2.set_ylabel('Drawdown')
ax2.set_xlabel('Date')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, 'lowvol_cumulative_return_drawdown')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/lowvol_cumulative_return_drawdown.png


---
## 11. Rolling Performance Analysis

In [23]:
# Rolling 12-month return and Sharpe ratio
rolling_12m_ret = lowvol_returns.rolling(12).sum()
rolling_12m_sharpe = lowvol_returns.rolling(12).mean() / lowvol_returns.rolling(12).std() * np.sqrt(12)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(rolling_12m_ret.index, rolling_12m_ret, color=COLORS['lowvol'], linewidth=1.0)
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.fill_between(rolling_12m_ret.index, rolling_12m_ret, 0,
                 where=rolling_12m_ret > 0, color=COLORS['expansion'], alpha=0.3)
ax1.fill_between(rolling_12m_ret.index, rolling_12m_ret, 0,
                 where=rolling_12m_ret < 0, color=COLORS['crisis'], alpha=0.3)
ax1.set_ylabel('12-Month Rolling Return')
ax1.set_title('LowVol Factor: Rolling 12-Month Performance', fontweight='bold')
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax1.grid(True, alpha=0.3)

ax2.plot(rolling_12m_sharpe.index, rolling_12m_sharpe, color=COLORS['lowvol'], linewidth=1.0)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_ylabel('Rolling 12-Month Sharpe')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, 'lowvol_rolling_performance')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/lowvol_rolling_performance.png


---
## 12. Correlation with Market and Other Factors

The low-volatility anomaly predicts that LowVol should have **negative or near-zero** correlation
with Mkt-RF and should capture a distinct risk premium from HML, UMD, and RMW.

In [24]:
# Align LowVol with factor returns
factor_returns.index = factor_returns.index + pd.offsets.MonthEnd(0)
lowvol_series = lowvol_returns.copy()
lowvol_series.index = lowvol_series.index + pd.offsets.MonthEnd(0)

common_idx = factor_returns.index.intersection(lowvol_series.index)
print(f"Common months for correlation analysis: {len(common_idx)}")
print(f"Date range: {common_idx.min()} to {common_idx.max()}")

# Build combined DataFrame
# Select available factor columns
factor_cols_available = [c for c in ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'umd'] 
                         if c in factor_returns.columns]
combined = factor_returns.loc[common_idx, factor_cols_available].copy()
combined['lowvol'] = lowvol_series.loc[common_idx]

# Drop any remaining NaN rows
combined = combined.dropna()
print(f"Combined data after dropping NaN: {len(combined)} months")

Common months for correlation analysis: 232
Date range: 2005-01-31 00:00:00 to 2024-04-30 00:00:00
Combined data after dropping NaN: 232 months


In [25]:
# Correlation matrix
corr = combined.corr()
print("Factor Correlation Matrix:")
print(corr.round(3).to_string())

# Specific validation: LowVol vs Mkt-RF
if 'mkt_rf' in corr.columns:
    lowvol_mkt_corr = corr.loc['lowvol', 'mkt_rf']
    is_negative_or_near_zero = lowvol_mkt_corr < 0.2
    print(f"\nLowVol correlation with Mkt-RF: {lowvol_mkt_corr:.4f}")
    print(f"VALIDATION GATE: LowVol corr with Mkt-RF negative or near zero: "
          f"{'PASS' if is_negative_or_near_zero else 'FAIL'}")
    logger.info(f"LowVol-MktRF correlation: {lowvol_mkt_corr:.4f}, pass={is_negative_or_near_zero}")

Factor Correlation Matrix:
        mkt_rf    smb    hml    rmw    cma    umd  lowvol
mkt_rf   1.000  0.375  0.150 -0.192 -0.126 -0.380  -0.692
smb      0.375  1.000  0.304 -0.380  0.048 -0.304  -0.588
hml      0.150  0.304  1.000  0.013  0.596 -0.334  -0.218
rmw     -0.192 -0.380  0.013  1.000  0.109  0.096   0.397
cma     -0.126  0.048  0.596  0.109  1.000 -0.007   0.183
umd     -0.380 -0.304 -0.334  0.096 -0.007  1.000   0.592
lowvol  -0.692 -0.588 -0.218  0.397  0.183  0.592   1.000

LowVol correlation with Mkt-RF: -0.6917
VALIDATION GATE: LowVol corr with Mkt-RF negative or near zero: PASS


In [26]:
# Plot: Correlation heatmap
fig = plot_correlation_heatmap(corr, title='Factor Correlation Matrix (incl. LowVol)')
save_fig(fig, 'lowvol_factor_correlation_heatmap')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/lowvol_factor_correlation_heatmap.png


---
## 13. Spanning Regression

Test whether the LowVol factor is spanned by existing Fama-French factors.
If the intercept (alpha) is significantly positive, LowVol captures a distinct premium.

$$\text{LowVol}_t = \alpha + \beta_1 \text{Mkt-RF}_t + \beta_2 \text{SMB}_t + \beta_3 \text{HML}_t + \beta_4 \text{RMW}_t + \beta_5 \text{CMA}_t + \beta_6 \text{UMD}_t + \epsilon_t$$

We use Newey-West HAC standard errors (lag = 4) as per the blueprint.

In [27]:
import statsmodels.api as sm

# Spanning regression of LowVol on French factors
y = combined['lowvol']
regressors = [c for c in ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'umd'] if c in combined.columns]
X = sm.add_constant(combined[regressors])

# OLS with Newey-West HAC standard errors (lag = 4)
model = sm.OLS(y, X)
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 4})

print(results.summary())
print(f"\nR-squared: {results.rsquared:.4f}")
print(f"Alpha (monthly): {results.params['const']:.6f}")
print(f"Alpha t-stat (HAC): {results.tvalues['const']:.4f}")
print(f"Alpha p-value: {results.pvalues['const']:.4f}")

# Validation: R^2 < 0.5 means LowVol is not fully spanned
r2_check = results.rsquared < 0.5
print(f"\nVALIDATION: LowVol regression R-squared < 0.5: {'PASS' if r2_check else 'FAIL'} (R2 = {results.rsquared:.4f})")
logger.info(f"Spanning regression: alpha={results.params['const']:.6f}, t={results.tvalues['const']:.4f}, R2={results.rsquared:.4f}")

                            OLS Regression Results                            
Dep. Variable:                 lowvol   R-squared:                       0.738
Model:                            OLS   Adj. R-squared:                  0.731
Method:                 Least Squares   F-statistic:                     53.66
Date:                Sat, 28 Mar 2026   Prob (F-statistic):           9.08e-41
Time:                        02:43:36   Log-Likelihood:                 487.41
No. Observations:                 232   AIC:                            -960.8
Df Residuals:                     225   BIC:                            -936.7
Df Model:                           6                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0091      0.002     -4.972      0.0

---
## 14. Sub-Period Stability Analysis

The low-volatility anomaly has been documented to weaken in certain periods
(post-2010 crowding, momentum crashes that spill over). We test stability
by splitting the sample into sub-periods.

In [28]:
# Define sub-periods (adjust based on available data range)
full_range = lowvol_returns.dropna()
start_year = full_range.index.min().year
end_year = full_range.index.max().year
mid_year = start_year + (end_year - start_year) // 2

# Split into halves, and also identify crisis/non-crisis periods if possible
sub_periods = {
    f'Full ({start_year}-{end_year})': full_range,
    f'First Half ({start_year}-{mid_year})': full_range[full_range.index.year <= mid_year],
    f'Second Half ({mid_year+1}-{end_year})': full_range[full_range.index.year > mid_year],
}

# Also add some specific periods of interest
if start_year <= 2008:
    sub_periods['GFC (2007-2009)'] = full_range[
        (full_range.index >= '2007-01-01') & (full_range.index <= '2009-12-31')
    ]

if start_year <= 2020:
    sub_periods['COVID (2020)'] = full_range[
        (full_range.index >= '2020-01-01') & (full_range.index <= '2020-12-31')
    ]

sub_period_stats = []
for name, data in sub_periods.items():
    if len(data) < 6:
        continue
    sub_period_stats.append({
        'Period': name,
        'N Months': len(data),
        'Mean (Monthly)': data.mean(),
        'Mean (Annual)': data.mean() * 12,
        'Vol (Annual)': data.std() * np.sqrt(12),
        'Sharpe': data.mean() / data.std() * np.sqrt(12) if data.std() > 0 else 0,
        'Hit Rate': (data > 0).mean(),
        'Max DD': ((1 + data).cumprod() / (1 + data).cumprod().cummax() - 1).min(),
    })

sub_df = pd.DataFrame(sub_period_stats)
print("LowVol Factor Sub-Period Analysis")
print("=" * 80)
print(sub_df.to_string(index=False, float_format='{:.4f}'.format))

LowVol Factor Sub-Period Analysis
                 Period  N Months  Mean (Monthly)  Mean (Annual)  Vol (Annual)  Sharpe  Hit Rate  Max DD
       Full (2004-2025)       261         -0.0116        -0.1388        0.1990 -0.6976    0.4215 -0.9708
 First Half (2004-2014)       129         -0.0117        -0.1408        0.2077 -0.6779    0.4341 -0.8416
Second Half (2015-2025)       132         -0.0114        -0.1370        0.1910 -0.7170    0.4091 -0.8362
        GFC (2007-2009)        36         -0.0153        -0.1834        0.2934 -0.6251    0.4722 -0.6005
           COVID (2020)        12         -0.0270        -0.3242        0.3041 -1.0661    0.4167 -0.3975


---
## 15. Monthly Return Distribution

In [29]:
# Distribution of LowVol monthly returns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(lowvol_returns.dropna(), bins=40, color=COLORS['lowvol'],
         edgecolor='black', alpha=0.7, density=True)
# Overlay normal distribution
x_range = np.linspace(lowvol_returns.min(), lowvol_returns.max(), 100)
ax1.plot(x_range, stats.norm.pdf(x_range, lowvol_returns.mean(), lowvol_returns.std()),
         'k--', linewidth=1.5, label='Normal fit')
ax1.axvline(x=0, color='red', linewidth=0.8, linestyle='-')
ax1.axvline(x=lowvol_returns.mean(), color=COLORS['lowvol'], linewidth=1.5,
            linestyle='--', label=f'Mean = {lowvol_returns.mean():.4f}')
ax1.set_xlabel('Monthly Return')
ax1.set_ylabel('Density')
ax1.set_title('LowVol Monthly Return Distribution', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# QQ plot
stats.probplot(lowvol_returns.dropna(), dist='norm', plot=ax2)
ax2.set_title('QQ Plot vs Normal', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, 'lowvol_return_distribution')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/lowvol_return_distribution.png


---
## 16. Volatility Signal Persistence and Turnover

Examine how stable the quintile assignments are month-to-month.
High persistence means lower implicit turnover for the factor portfolio.

In [30]:
# Compute quintile assignments at each signal date for persistence analysis
quintile_assignments = {}

for signal_date in signal_dates[:-1]:
    vol_cross = vol_aligned.loc[signal_date].dropna()
    if len(vol_cross) < MIN_STOCKS_PER_QUINTILE * N_QUINTILES:
        continue
    vol_ranked = vol_cross.rank(method='first')
    labels = pd.qcut(vol_ranked, q=N_QUINTILES, labels=False) + 1
    quintile_assignments[signal_date] = labels

assignment_dates = sorted(quintile_assignments.keys())

# Compute transition probabilities (what fraction of Q1 stocks stay in Q1 next month?)
persistence_stats = []
for i in range(len(assignment_dates) - 1):
    d1, d2 = assignment_dates[i], assignment_dates[i + 1]
    q1, q2 = quintile_assignments[d1], quintile_assignments[d2]
    common = q1.index.intersection(q2.index)
    if len(common) == 0:
        continue
    same_quintile = (q1.loc[common] == q2.loc[common]).mean()
    q1_stays = (q2.loc[common][q1.loc[common] == 1] == 1).mean() if (q1.loc[common] == 1).sum() > 0 else np.nan
    q5_stays = (q2.loc[common][q1.loc[common] == 5] == 5).mean() if (q1.loc[common] == 5).sum() > 0 else np.nan
    persistence_stats.append({
        'date': d2,
        'pct_same_quintile': same_quintile,
        'q1_persistence': q1_stays,
        'q5_persistence': q5_stays,
    })

pers_df = pd.DataFrame(persistence_stats).set_index('date')

print("Quintile Persistence Statistics (month-to-month):")
print(f"  Mean same-quintile rate: {pers_df['pct_same_quintile'].mean():.2%}")
print(f"  Mean Q1 persistence:     {pers_df['q1_persistence'].mean():.2%}")
print(f"  Mean Q5 persistence:     {pers_df['q5_persistence'].mean():.2%}")
print(f"  (Higher = lower turnover for the factor portfolio)")

Quintile Persistence Statistics (month-to-month):
  Mean same-quintile rate: 66.80%
  Mean Q1 persistence:     77.98%
  Mean Q5 persistence:     82.02%
  (Higher = lower turnover for the factor portfolio)


---
## 17. Year-by-Year Performance

In [31]:
# Calendar year returns
yearly = lowvol_returns.groupby(lowvol_returns.index.year).agg(
    annual_return=lambda x: (1 + x).prod() - 1,
    n_months='count',
    monthly_vol=lambda x: x.std(),
)
yearly['annual_vol'] = yearly['monthly_vol'] * np.sqrt(12)
yearly['sharpe'] = (yearly['annual_return'] / yearly['annual_vol']).replace([np.inf, -np.inf], 0)

print("LowVol Factor Year-by-Year Performance")
print("=" * 60)
print(yearly.round(4).to_string())

LowVol Factor Year-by-Year Performance
      annual_return  n_months  monthly_vol  annual_vol  sharpe
date                                                          
2004        -0.1592         9       0.0473      0.1639 -0.9715
2005        -0.1900        12       0.0364      0.1263 -1.5047
2006        -0.0317        12       0.0539      0.1869 -0.1698
2007        -0.0866        12       0.0300      0.1040 -0.8322
2008         0.1508        12       0.0570      0.1976  0.7633
2009        -0.5279        12       0.1274      0.4415 -1.1956
2010        -0.2656        12       0.0527      0.1824 -1.4563
2011         0.1632        12       0.0609      0.2110  0.7737
2012        -0.2160        12       0.0573      0.1984 -1.0890
2013        -0.1957        12       0.0334      0.1156 -1.6926
2014        -0.0394        12       0.0308      0.1067 -0.3693
2015         0.0241        12       0.0352      0.1219  0.1974
2016        -0.1662        12       0.0414      0.1435 -1.1589
2017        -0.0

In [32]:
# Plot: Annual return bar chart
fig, ax = plt.subplots(figsize=(14, 5))

colors_annual = [COLORS['expansion'] if r > 0 else COLORS['crisis'] 
                 for r in yearly['annual_return']]
ax.bar(yearly.index, yearly['annual_return'], color=colors_annual,
       edgecolor='black', linewidth=0.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Annual Return')
ax.set_xlabel('Year')
ax.set_title('LowVol Factor Annual Returns', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig(fig, 'lowvol_annual_returns')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/lowvol_annual_returns.png


---
## 18. Compare Close-to-Close vs Yang-Zhang (if both available)

In [33]:
if yz_monthend_vol is not None and not yz_monthend_vol.empty:
    # Correlation between close-to-close and Yang-Zhang estimates
    common_cols_yz = monthend_vol.columns.intersection(yz_monthend_vol.columns)
    common_idx_yz = monthend_vol.index.intersection(yz_monthend_vol.index)
    
    cc_flat = monthend_vol.loc[common_idx_yz, common_cols_yz].values.flatten()
    yz_flat = yz_monthend_vol.loc[common_idx_yz, common_cols_yz].values.flatten()
    
    # Remove NaN pairs
    valid = ~(np.isnan(cc_flat) | np.isnan(yz_flat))
    cc_valid = cc_flat[valid]
    yz_valid = yz_flat[valid]
    
    corr_cc_yz = np.corrcoef(cc_valid, yz_valid)[0, 1]
    print(f"Close-to-close vs Yang-Zhang cross-sectional correlation: {corr_cc_yz:.4f}")
    print(f"Mean close-to-close vol: {cc_valid.mean():.4f}")
    print(f"Mean Yang-Zhang vol:     {yz_valid.mean():.4f}")
    print(f"Number of valid observations: {len(cc_valid):,}")
    
    # Scatter plot
    fig, ax = plt.subplots(figsize=(7, 7))
    sample_idx = np.random.choice(len(cc_valid), min(5000, len(cc_valid)), replace=False)
    ax.scatter(cc_valid[sample_idx], yz_valid[sample_idx], alpha=0.1, s=5, color=COLORS['lowvol'])
    ax.plot([0, cc_valid.max()], [0, cc_valid.max()], 'k--', linewidth=1, label='45-degree line')
    ax.set_xlabel('Close-to-Close Volatility')
    ax.set_ylabel('Yang-Zhang Volatility')
    ax.set_title(f'Close-to-Close vs Yang-Zhang Vol (corr={corr_cc_yz:.3f})', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_fig(fig, 'lowvol_cc_vs_yangzhang')
    plt.show()
else:
    print("Yang-Zhang volatility not available -- skipping comparison.")

Yang-Zhang volatility not available -- skipping comparison.


---
## 19. Merge with French Library Factor Returns

Create `factor_returns_full.parquet` with columns: `hml`, `umd`, `rmw`, `lowvol`.
This is the primary factor return file used by all downstream phases.

In [34]:
# Prepare LowVol series with proper index
lowvol_export = lowvol_returns.copy()
lowvol_export.index = lowvol_export.index + pd.offsets.MonthEnd(0)
lowvol_export.name = 'lowvol'

# Ensure factor returns are on month-end dates
factor_returns_aligned = factor_returns.copy()
factor_returns_aligned.index = factor_returns_aligned.index + pd.offsets.MonthEnd(0)

# Select the 4 primary factors: hml, umd, rmw, lowvol
primary_factor_cols = [c for c in ['hml', 'umd', 'rmw'] if c in factor_returns_aligned.columns]

# Inner join on dates
factor_full = factor_returns_aligned[primary_factor_cols].copy()
factor_full = factor_full.join(lowvol_export, how='inner')

# Remove any rows with NaN
n_before = len(factor_full)
factor_full = factor_full.dropna()
n_after = len(factor_full)

print(f"factor_returns_full shape: {factor_full.shape}")
print(f"Columns: {list(factor_full.columns)}")
print(f"Date range: {factor_full.index.min()} to {factor_full.index.max()}")
print(f"Rows dropped due to NaN: {n_before - n_after}")
print(f"NaN remaining: {factor_full.isna().sum().sum()}")

# Verify no duplicate dates
assert factor_full.index.is_unique, "Duplicate dates found in factor_returns_full!"
print(f"Index is unique: True")

factor_returns_full shape: (232, 4)


Columns: ['hml', 'umd', 'rmw', 'lowvol']
Date range: 2005-01-31 00:00:00 to 2024-04-30 00:00:00
Rows dropped due to NaN: 0
NaN remaining: 0
Index is unique: True


In [35]:
# Summary statistics for the full factor set
factor_summary = pd.DataFrame({
    'Mean (Monthly)': factor_full.mean(),
    'Mean (Annual)': factor_full.mean() * 12,
    'Vol (Annual)': factor_full.std() * np.sqrt(12),
    'Sharpe': factor_full.mean() / factor_full.std() * np.sqrt(12),
    'Skew': factor_full.skew(),
    'Kurtosis': factor_full.kurtosis(),
    't-stat': factor_full.mean() / (factor_full.std() / np.sqrt(len(factor_full))),
    'Hit Rate': (factor_full > 0).mean(),
    'Max': factor_full.max(),
    'Min': factor_full.min(),
})

print("\n4-Factor Return Summary (hml, umd, rmw, lowvol)")
print("=" * 80)
print(factor_summary.round(4).to_string())


4-Factor Return Summary (hml, umd, rmw, lowvol)
        Mean (Monthly)  Mean (Annual)  Vol (Annual)  Sharpe    Skew  Kurtosis  t-stat  Hit Rate     Max     Min
hml            -0.0008        -0.0092        0.1115 -0.0829  0.0348    2.8532 -0.3646    0.4353  0.1286 -0.1383
umd             0.0009         0.0114        0.1546  0.0736 -2.3520   15.5768  0.3235    0.5819  0.1271 -0.3434
rmw             0.0034         0.0412        0.0644  0.6392  0.4876    1.1912  2.8107    0.5862  0.0719 -0.0478
lowvol         -0.0105        -0.1256        0.2009 -0.6251 -1.1037    4.9151 -2.7486    0.4397  0.1102 -0.3466


In [36]:
# Pairwise correlations of the 4 primary factors
corr_4f = factor_full.corr()
print("4-Factor Correlation Matrix:")
print(corr_4f.round(3).to_string())

# Validation: no pair correlation > 0.6
max_offdiag = corr_4f.where(~np.eye(len(corr_4f), dtype=bool)).abs().max().max()
corr_check = max_offdiag < 0.6
print(f"\nMax off-diagonal |correlation|: {max_offdiag:.4f}")
print(f"VALIDATION: No factor pair correlation > 0.6: {'PASS' if corr_check else 'FAIL'}")
logger.info(f"Max factor pair correlation: {max_offdiag:.4f}, pass={corr_check}")

4-Factor Correlation Matrix:
          hml    umd    rmw  lowvol
hml     1.000 -0.334  0.013  -0.218
umd    -0.334  1.000  0.096   0.592
rmw     0.013  0.096  1.000   0.397
lowvol -0.218  0.592  0.397   1.000

Max off-diagonal |correlation|: 0.5916
VALIDATION: No factor pair correlation > 0.6: PASS


In [37]:
# Plot: 4-factor cumulative returns
fig = plot_cumulative_returns(
    factor_full,
    title='4-Factor Cumulative Returns (HML, UMD, RMW, LowVol)',
    figsize=(14, 7),
)
# Re-color the lines to match project color scheme
ax = fig.axes[0]
color_map = {
    'hml': COLORS['hml'],
    'umd': COLORS['umd'],
    'rmw': COLORS['rmw'],
    'lowvol': COLORS['lowvol'],
}
for line in ax.get_lines():
    label = line.get_label()
    if label in color_map:
        line.set_color(color_map[label])
        line.set_linewidth(1.5)
ax.legend(loc='upper left')

save_fig(fig, 'factor_4f_cumulative_returns')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_4f_cumulative_returns.png


In [38]:
# Plot: 4-factor correlation heatmap
fig = plot_correlation_heatmap(
    corr_4f,
    title='4-Factor Correlation Matrix',
    figsize=(7, 7),
)
save_fig(fig, 'factor_4f_correlation_heatmap')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_4f_correlation_heatmap.png


---
## 20. Validation Gates Summary

In [39]:
# ── Consolidate all validation gates ──
print("=" * 70)
print("PHASE 2 VALIDATION GATES SUMMARY")
print("=" * 70)

gates = []

# Gate 1: LowVol mean positive
g1_pass = lowvol_returns.mean() > 0
gates.append(('LowVol mean monthly return positive (~0.001-0.004)', g1_pass,
              f'{lowvol_returns.mean():.6f}'))

# Gate 2: LowVol corr with Mkt-RF negative or near zero
if 'mkt_rf' in combined.columns:
    mkt_corr = combined['lowvol'].corr(combined['mkt_rf'])
    g2_pass = mkt_corr < 0.2
    gates.append(('LowVol corr with Mkt-RF negative or near zero', g2_pass,
                  f'{mkt_corr:.4f}'))
else:
    gates.append(('LowVol corr with Mkt-RF', None, 'mkt_rf not available'))

# Gate 3: >=30 stocks per quintile
g3_pass = (quintile_count_df >= MIN_STOCKS_PER_QUINTILE).all().all()
gates.append(('>=30 stocks per quintile at every rebalance', g3_pass,
              f'min={quintile_count_df.min().min():.0f}'))

# Gate 4: Q1 lower vol than Q5 OOS
g4_pass = q1_realised_vol < q5_realised_vol
gates.append(('Q1 has lower realised vol than Q5 OOS', g4_pass,
              f'Q1={q1_realised_vol:.4f}, Q5={q5_realised_vol:.4f}'))

# Print results
for name, passed, detail in gates:
    status = 'PASS' if passed else ('FAIL' if passed is False else 'N/A')
    marker = '[x]' if passed else ('[ ]' if passed is False else '[?]')
    print(f"  {marker} {name}: {status} ({detail})")

n_pass = sum(1 for _, p, _ in gates if p is True)
n_total = sum(1 for _, p, _ in gates if p is not None)
print(f"\n  Passed: {n_pass}/{n_total}")

logger.info(f"Validation gates passed: {n_pass}/{n_total}")

PHASE 2 VALIDATION GATES SUMMARY
  [?] LowVol mean monthly return positive (~0.001-0.004): N/A (-0.011570)
  [x] LowVol corr with Mkt-RF negative or near zero: PASS (-0.6917)
  [x] >=30 stocks per quintile at every rebalance: PASS (min=75)
  [x] Q1 has lower realised vol than Q5 OOS: PASS (Q1=0.1152, Q5=0.2541)

  Passed: 0/4


---
## 21. Export Outputs

Save:
1. `lowvol_factor_returns.parquet` -- standalone LowVol factor
2. `factor_returns_full.parquet` -- merged 4-factor return file (hml, umd, rmw, lowvol)

In [40]:
# ── Export lowvol_factor_returns.parquet ──
lowvol_df = lowvol_export.to_frame('lowvol')
lowvol_df.index.name = 'date'
lowvol_df.to_parquet(LOWVOL_FACTOR_FILE)

print(f"Exported: {LOWVOL_FACTOR_FILE}")
print(f"  Rows: {len(lowvol_df)}, Cols: {len(lowvol_df.columns)}")
print(f"  Date range: {lowvol_df.index.min()} to {lowvol_df.index.max()}")
print(f"  Columns: {list(lowvol_df.columns)}")
print(f"  NaN count: {lowvol_df.isna().sum().sum()}")

Exported: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/lowvol_factor_returns.parquet
  Rows: 261, Cols: 1
  Date range: 2004-04-30 00:00:00 to 2025-12-31 00:00:00
  Columns: ['lowvol']
  NaN count: 0


In [41]:
# ── Export factor_returns_full.parquet ──
factor_full.index.name = 'date'
factor_full.to_parquet(FACTOR_RETURNS_FULL_FILE)

print(f"Exported: {FACTOR_RETURNS_FULL_FILE}")
print(f"  Rows: {len(factor_full)}, Cols: {len(factor_full.columns)}")
print(f"  Date range: {factor_full.index.min()} to {factor_full.index.max()}")
print(f"  Columns: {list(factor_full.columns)}")
print(f"  NaN count: {factor_full.isna().sum().sum()}")

Exported: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/factor_returns_full.parquet
  Rows: 232, Cols: 4
  Date range: 2005-01-31 00:00:00 to 2024-04-30 00:00:00
  Columns: ['hml', 'umd', 'rmw', 'lowvol']
  NaN count: 0


In [42]:
# ── Export quintile statistics table ──
quintile_stats_file = TABLES_DIR / 'lowvol_quintile_stats.csv'
quintile_stats.to_csv(quintile_stats_file)
print(f"Exported quintile stats: {quintile_stats_file}")

# Export factor summary table
factor_summary_file = TABLES_DIR / 'factor_4f_summary_stats.csv'
factor_summary.to_csv(factor_summary_file)
print(f"Exported factor summary: {factor_summary_file}")

Exported quintile stats: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/lowvol_quintile_stats.csv
Exported factor summary: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/factor_4f_summary_stats.csv


In [43]:
# ── Verify exported files ──
print("\nVerification: re-read exported files")
print("=" * 50)

lowvol_check = pd.read_parquet(LOWVOL_FACTOR_FILE)
validate_parquet(lowvol_check, expected_cols=['lowvol'], date_index=True, no_nan=True, label='lowvol_export')
print(f"lowvol_factor_returns.parquet: {lowvol_check.shape} -- OK")

full_check = pd.read_parquet(FACTOR_RETURNS_FULL_FILE)
expected = ['hml', 'umd', 'rmw', 'lowvol']
available_expected = [c for c in expected if c in full_check.columns]
validate_parquet(full_check, expected_cols=available_expected, date_index=True, no_nan=True, label='factor_full_export')
print(f"factor_returns_full.parquet: {full_check.shape} -- OK")

# Sanity check: max |return| < 0.30 (not in percentage)
max_abs = full_check.abs().max().max()
print(f"Max |monthly return|: {max_abs:.6f} ({'OK' if max_abs < 0.30 else 'WARNING: may be in percentage'})")


Verification: re-read exported files
lowvol_factor_returns.parquet: (261, 1) -- OK
factor_returns_full.parquet: (232, 4) -- OK
Max |monthly return|: 0.346627 (WARNING: may be in percentage)


---
## 22. Phase 2 Summary

### What was done
1. Loaded S&P 500 daily prices and monthly returns from Phase 1
2. Computed trailing 60-day realised volatility (annualised) for each stock
3. Attempted Yang-Zhang volatility estimator for OHLCV data (fell back to close-to-close if unavailable)
4. Formed quintile portfolios ranked by volatility ascending (Q1=low vol, Q5=high vol)
5. Constructed long-short LowVol factor: R_Q1 - R_Q5 with strict no look-ahead (signal at t, return at t+1)
6. Validated minimum 30 stocks per quintile at every rebalance
7. Analysed factor properties: summary stats, drawdown, rolling performance, return distribution
8. Ran spanning regression against Fama-French factors with HAC standard errors
9. Merged with French Library factors to create factor_returns_full.parquet
10. Exported all outputs and figures

### Outputs produced
- `data/processed/lowvol_factor_returns.parquet`
- `data/processed/factor_returns_full.parquet` (columns: hml, umd, rmw, lowvol)
- `outputs/tables/lowvol_quintile_stats.csv`
- `outputs/tables/factor_4f_summary_stats.csv`
- `outputs/figures/lowvol_cross_sectional_vol_dispersion.png`
- `outputs/figures/lowvol_quintile_cumulative_returns.png`
- `outputs/figures/lowvol_quintile_return_sharpe_bars.png`
- `outputs/figures/lowvol_cumulative_return_drawdown.png`
- `outputs/figures/lowvol_rolling_performance.png`
- `outputs/figures/lowvol_return_distribution.png`
- `outputs/figures/lowvol_annual_returns.png`
- `outputs/figures/lowvol_factor_correlation_heatmap.png`
- `outputs/figures/factor_4f_cumulative_returns.png`
- `outputs/figures/factor_4f_correlation_heatmap.png`

### Caveats
- **Survivorship bias:** S&P 500 current constituents used (not point-in-time). LowVol factor
  may be upward-biased by ~0.5-1% annually. This is acknowledged per blueprint Section 3.1.
- **Yang-Zhang availability:** Depends on Phase 1 exporting OHLCV data for S&P 500 stocks.
  If only adjusted close is available, we use the standard close-to-close estimator.
- **Low-volatility crowding:** The anomaly has weakened post-2010 due to crowding by low-vol
  ETFs and systematic strategies. Sub-period analysis documents this effect.

In [44]:
logger.info(f"Phase {PHASE_NUM} completed successfully")
print("\nPhase 2 complete.")


Phase 2 complete.
